# add-sub-div-back-lambdas — faded example 2: Complete the size-1-axis summation in unbroadcast

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `add-sub-div-back-lambdas`. Running the beacon reports progress on the `Backprop: add/sub/div back as lambdas` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: add/sub/div back as lambdas` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`add-sub-div-back-lambdas`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "add-sub-div-back-lambdas"
DD_SUBTOPIC = "Backprop: add/sub/div back as lambdas"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`unbroadcast(grad, target)` reduces a gradient computed at the broadcast shape back to `target.shape`. After dropping extra leading axes, any axis where `target` had size 1 must be summed (keepdim) because broadcasting copied the input along that axis, and gradients from copies add. Forgetting this returns the wrong shape.

## Faded exercise 2

### Finish unbroadcast

The leading-axis reduction is filled in. Complete the loop body that collapses each size-1 axis of `target` by summing `grad` over that axis with `keepdim=True`, so the result has shape exactly equal to `target.shape`.

**Fill in:** for each axis where target has size 1, sum grad over that axis with keepdim=True

In [ ]:
def unbroadcast(grad, target):
    while grad.ndim > target.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(target.shape):
        if size == 1:
            raise NotImplementedError()  # TODO: for each axis where target has size 1, sum grad over that axis with keepdim=True
    return grad


def _test():
    t.manual_seed(2)
    x = t.randn(1, 4, requires_grad=True)   # broadcasts over rows
    y = t.randn(3, 4, requires_grad=True)
    out = x - y                              # shape (3, 4)
    g = t.randn(3, 4)
    out.backward(g)
    # sub arg0 backward is g, then unbroadcast to x.shape
    dx = unbroadcast(g, x.detach())
    assert tuple(dx.shape) == (1, 4)
    assert t.allclose(dx, x.grad, atol=1e-5)
    # also check a leading-axis case: target (4,) vs out (2,4)
    z = t.randn(4, requires_grad=True)
    w = t.randn(2, 4, requires_grad=True)
    o2 = z + w
    g2 = t.randn(2, 4)
    o2.backward(g2)
    dz = unbroadcast(g2, z.detach())
    assert tuple(dz.shape) == (4,)
    assert t.allclose(dz, z.grad, atol=1e-5)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def unbroadcast(grad, target):
    while grad.ndim > target.ndim:
        grad = grad.sum(dim=0)
    for i, size in enumerate(target.shape):
        if size == 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad
```
</details>